In [ ]:
import os
import google.generativeai as genai
from duckduckgo_search import DDGS

genai.configure(api_key=api_key)


def calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


# tool - web search
def web_search(query: str) -> str:
    try:
        with DDGS() as ddgs:
            results =[r for r in ddgs.text(query, max_results=3)]
        summaries = "\n".join([f"- {res['title']}: {res['body']}" for res in results])
        return f"Top search results:\n{summaries}"
    except Exception as e:
        return f"Error: {str(e)}" 

model = genai.GenerativeModel("gemini-2.5-flash", tools=[calculator, web_search])



c:\Users\raman\OneDrive\Desktop\Documents\75 day AI engineer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#agent's conversation tool

chat = model.start_chat(history=[])  # chat history

def agent_response(user_input:str)->str:
    response = chat.send_message(user_input)
    
    if response.candidates[0].content.parts and response.candidates[0].content.parts[0].function_call:
        func_call = response.candidates[0].content.parts[0].function_call
        func_name = func_call.name
        args = dict(func_call.args)
        tool_result = None
        if func_name =="calculator":
            tool_result = calculator(args.get("expression", "")) # generate natural language
        elif func_name =="web_search":
            tool_result = web_search(args.get("query", ""))
        if tool_result is None:
            tool_result = "Error: Tool execution failed."
            return tool_result
        else:
            final_response = chat.send_message(genai.protos.Part(
                function_response=genai.protos.FunctionResponse(
                    name=func_name,
                    response={'result':tool_result}
                )
            ))
            return final_response.text
    else:
        return response.text
        

In [ ]:
print("Student Helper Agent: Ask me anything! Type 'exit' to quit.")

while True:
    user_input = input("You: ")
    if user_input.lower() == 'exit':
        print("Student Helper Agent: Goodbye!")
        break
    response = agent_response(user_input)
    print("Student Helper Agent:", response)

Student Helper Agent: Ask me anything! Type 'exit' to quit.


C:\Users\raman\AppData\Local\Temp\ipykernel_30380\1953832843.py:27: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Student Helper Agent: I'm sorry, I cannot answer your question. My search results were not helpful.


C:\Users\raman\AppData\Local\Temp\ipykernel_30380\1953832843.py:27: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Student Helper Agent: Narendra Modi is the current Prime Minister of India. He was sworn in for his third term on June 9, 2024, after a victory in the 2024 Parliamentary elections.
